In [1]:
import nflreadpy as nfl
import pandas as pd
import json

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

sched = nfl.load_schedules(YEARS).to_pandas()
sched = sched[sched['game_type'] == 'REG'].copy()

# Keep only what we need
cols = ['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']
sched = sched[cols].dropna(subset=['home_score', 'away_score'])
sched['home_score'] = sched['home_score'].astype(int)
sched['away_score'] = sched['away_score'].astype(int)

print(f"Games: {len(sched)}")
print(f"Seasons: {sorted(sched['season'].unique())}")
print(sched.head())

with open('schedule_data.json', 'w') as f:
    json.dump(sched.to_dict(orient='records'), f)
print("Saved schedule_data.json")

Games: 1615
Seasons: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
   season  week home_team away_team  home_score  away_score
0    2020     1        KC       HOU          34          20
1    2020     1       ATL       SEA          25          38
2    2020     1       BAL       CLE          38           6
3    2020     1       BUF       NYJ          27          17
4    2020     1       CAR        LV          30          34
Saved schedule_data.json


In [1]:
import nflreadpy as nfl
import pandas as pd
import json

seasons = [2020, 2021, 2022, 2023, 2024, 2025]

# ── 1. Load player stats ──────────────────────────────────────────────────
stats = nfl.load_player_stats(seasons).to_pandas()
stats = stats[stats['season_type'] == 'REG']

# ── 2. Offensive rushing stats (from RBs, QBs, WRs — all rushers) ─────────
rush_off = stats.groupby(['season', 'team']).agg(
    rush_yards=('rushing_yards', 'sum'),
    rush_tds=('rushing_tds', 'sum'),
    rush_carries=('carries', 'sum')
).reset_index()

# ── 3. Offensive passing stats ────────────────────────────────────────────
pass_off = stats[stats['position'] == 'QB'].groupby(['season', 'team']).agg(
    pass_yards=('passing_yards', 'sum'),
    pass_tds=('passing_tds', 'sum'),
    pass_attempts=('attempts', 'sum'),
    completions=('completions', 'sum'),
    interceptions=('passing_interceptions', 'sum'),
    pass_epa=('passing_epa', 'sum')
).reset_index()

# ── 4. Receiving stats allowed (use opponent_team as the DEFENSE) ─────────
# Rushing yards ALLOWED = what opponents rushed against this team
rush_def = stats.groupby(['season', 'opponent_team']).agg(
    rush_yards_allowed=('rushing_yards', 'sum'),
    rush_tds_allowed=('rushing_tds', 'sum')
).reset_index().rename(columns={'opponent_team': 'team'})

# Passing yards ALLOWED
pass_def = stats[stats['position'] == 'QB'].groupby(['season', 'opponent_team']).agg(
    pass_yards_allowed=('passing_yards', 'sum'),
    pass_tds_allowed=('passing_tds', 'sum'),
    interceptions_forced=('passing_interceptions', 'sum')
).reset_index().rename(columns={'opponent_team': 'team'})

# ── 5. Scores from schedule (points for / against) ───────────────────────
sched = nfl.load_schedules(seasons).to_pandas()
sched = sched[sched['game_type'] == 'REG'][['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].dropna()

home = sched[['season', 'week', 'home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'points_for', 'away_score': 'points_against'})
away = sched[['season', 'week', 'away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'points_for', 'home_score': 'points_against'})

all_games = pd.concat([home, away])
scoring = all_games.groupby(['season', 'team']).agg(
    points_for=('points_for', 'sum'),
    points_against=('points_against', 'sum'),
    games_played=('week', 'count')
).reset_index()

# ── 6. Merge everything ───────────────────────────────────────────────────
team_stats = scoring.copy()
team_stats = team_stats.merge(rush_off, on=['season', 'team'], how='left')
team_stats = team_stats.merge(pass_off, on=['season', 'team'], how='left')
team_stats = team_stats.merge(rush_def, on=['season', 'team'], how='left')
team_stats = team_stats.merge(pass_def, on=['season', 'team'], how='left')

# ── 7. Per-game rates ─────────────────────────────────────────────────────
gp = team_stats['games_played']
team_stats['rush_ypg']          = (team_stats['rush_yards']          / gp).round(1)
team_stats['rush_ypg_allowed']  = (team_stats['rush_yards_allowed']  / gp).round(1)
team_stats['pass_ypg']          = (team_stats['pass_yards']          / gp).round(1)
team_stats['pass_ypg_allowed']  = (team_stats['pass_yards_allowed']  / gp).round(1)
team_stats['ppg']               = (team_stats['points_for']          / gp).round(1)
team_stats['ppg_against']       = (team_stats['points_against']      / gp).round(1)
team_stats['completion_pct']    = (team_stats['completions'] / team_stats['pass_attempts'] * 100).round(1)

# ── 8. Fill NaN and export ────────────────────────────────────────────────
team_stats = team_stats.fillna(0)

# Round integers
int_cols = ['rush_yards','rush_tds','rush_carries','pass_yards','pass_tds',
            'pass_attempts','completions','interceptions','rush_yards_allowed',
            'rush_tds_allowed','pass_yards_allowed','pass_tds_allowed',
            'interceptions_forced','points_for','points_against','games_played']
for col in int_cols:
    if col in team_stats.columns:
        team_stats[col] = team_stats[col].astype(int)

output = team_stats.to_dict(orient='records')
with open('team_stats_data.json', 'w') as f:
    json.dump(output, f)

print(f"✅ Exported {len(output)} team-season records")
print(team_stats[team_stats['season'] == 2025][['team','ppg','rush_ypg','pass_ypg']].sort_values('ppg', ascending=False).head(10))

✅ Exported 192 team-season records
    team   ppg  rush_ypg  pass_ypg
176   LA  30.5     126.6     276.9
181   NE  28.8     128.9     262.3
187  SEA  28.4     123.3     239.0
170  DET  28.3     120.1     268.5
163  BUF  28.3     159.6     234.2
174  JAX  27.9     115.1     236.8
168  DAL  27.7     125.6     278.5
173  IND  27.4     118.1     239.4
165  CHI  25.9     144.2     234.6
188   SF  25.7     106.9     254.0
